In [24]:
import os
from databricks.connect import DatabricksSession

# Initialize Spark locally when needed
if "spark" not in globals():
    spark = DatabricksSession.builder.serverless().getOrCreate()

#NotebookParameters
if "dbutils" in globals():
    #Execute in Databricks
    dbutils.widgets.dropdown(
        "environment", 
        "dev", 
        ["dev", "staging", "prod"]
    )

    dbutils.widgets.text("organization", "databricks")

    environment = dbutils.widgets.get("environment")
    organization = dbutils.widgets.get("organization").strip()
else:
    #Execute in VSCode with Databricks Connect
    environment = os.getenv("LAKEHOUSE_ENV", "dev")
    organization = os.getenv("GITHUB_ORGANIZATION", "databricks").strip()

catalogs = {
    "dev": "opensource_dev",
    "staging": "opensource_staging",
    "prod": "opensource_prod"
}

catalog = catalogs[environment]
schema_name = "github_analytics"

changes_table = f"{catalog}.{schema_name}.silver_changes_github_repositories"
silver_table = f"{catalog}.{schema_name}.silver_github_repositories"

print(f"Source Table: {changes_table}")
print(f"Target Table: {silver_table}")

Source Table: opensource_dev.github_analytics.silver_changes_github_repositories
Target Table: opensource_dev.github_analytics.silver_github_repositories


In [25]:
from pyspark.sql import functions as F

#Read and updated repositories from Change Directories
df = (
    spark.read.table(changes_table)
    .filter(F.col("organization") == organization)
)

print(f"Repositories to process: {df.count()}")

df.select(
    "id",
    "full_name",
    "description",
    "language",
    "change_type"
).show(10, truncate=False)

Repositories to process: 0
+---+---------+-----------+--------+-----------+
|id |full_name|description|language|change_type|
+---+---------+-----------+--------+-----------+
+---+---------+-----------+--------+-----------+



In [26]:
df_cleaned = (
    df
    .withColumn(
        "name",
        F.trim(F.col("name"))
    )
    .withColumn(
        "full_name",
        F.trim(F.col("full_name"))
    )
    .withColumn(
        "description",
        F.trim(F.col("description"))
    )
    .withColumn(
        "language",
        F.trim(F.col("language"))
    )
    .select(
        "id",
        "name",
        "full_name",
        "description",
        "language",
        "archived",
        "default_branch",
        "stargazers_count",
        "forks_count",
        "organization",
        "snapshot_id",
        "ingestion_ts",
        "change_type"
    )
)

df_cleaned.show(10, truncate=False)

+---+----+---------+-----------+--------+--------+--------------+----------------+-----------+------------+-----------+------------+-----------+
|id |name|full_name|description|language|archived|default_branch|stargazers_count|forks_count|organization|snapshot_id|ingestion_ts|change_type|
+---+----+---------+-----------+--------+--------+--------------+----------------+-----------+------------+-----------+------------+-----------+
+---+----+---------+-----------+--------+--------+--------------+----------------+-----------+------------+-----------+------------+-----------+



In [27]:
#Check mandatory fields
invalid_record = df_cleaned.filter(
    F.col("id").isNull() |
    F.col("full_name").isNull() |
    F.col("snapshot_id").isNull() |
    F.col("ingestion_ts").isNull() 
)

#Check duplicate repository IDs within the same snapshot
duplicate_records = (
    df_cleaned
    .groupBy("id", "snapshot_id")
    .count()
    .filter(F.col("count") > 1)
)

invalid_count = invalid_record.count()
duplicate_count = duplicate_records.count()

print(f"Invalid records: {invalid_count}")
print(f"Duplicate repository keys: {duplicate_count}")

if invalid_count > 0 or duplicate_count > 0:
    raise ValueError("Silver data quality validation failed")

Invalid records: 0
Duplicate repository keys: 0


In [28]:

# Initial load
if not spark.catalog.tableExists(silver_table):
    (
        df_cleaned.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    inserted_records = df_cleaned.count()

else:
    # Retrieve already processed repository keys
    existing_df = (
        spark.read.table(silver_table)
        .select("id", "snapshot_id")
    )

    # Identify records not yet loaded
    new_records = (
        df_cleaned
        .join(
            existing_df,
            on=["id", "snapshot_id"],
            how="left_anti"
        )
    )

    inserted_records = new_records.count()

    # Append only new records
    if inserted_records > 0:
        (
            new_records.write
            .format("delta")
            .mode("append")
            .saveAsTable(silver_table)
        )

print(f"Silver table: {silver_table}")
print(f"Records inserted: {inserted_records}")

Silver table: opensource_dev.github_analytics.silver_github_repositories
Records inserted: 0


In [29]:
#Verify the Silver Table
df_check = spark.read.table(silver_table)

print(f"Total Silver records: {df_check.count()}")

df_check.groupBy("change_type").count().show()

#Check duplicate repository keys
(
    df_check.groupBy("id", "snapshot_id")  
    .count() 
    .filter(F.col("count") > 1)
    .show()
)

Total Silver records: 253
+-----------+-----+
|change_type|count|
+-----------+-----+
|        NEW|  253|
+-----------+-----+

+---+-----------+-----+
| id|snapshot_id|count|
+---+-----------+-----+
+---+-----------+-----+

